In [ ]:
# ==============================================================================
# CELL 1: INSTALASI SEMUA LIBRARY YANG DIGUNAKAN
# ==============================================================================
!pip install pandas numpy scikit-learn nltk seaborn matplotlib

# Analisis Sentimen, Ekstraksi Fitur, Klasifikasi, dan Clustering Dokumen
## Dataset: IMDb Dataset of 50K Movie Reviews
**Sumber Dataset:** [Kaggle - IMDb Dataset of 50K Movie Reviews](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews)

Notebook ini mencakup alur lengkap analisis teks ulasan film:
1. **Instalasi Dependensi**: Instalasi pustaka (Cell 1 di atas).
2. **Import Library & Environment**: Konfigurasi NLP, Scikit-Learn, dan visualisasi.
3. **Data Loading & Preprocessing**: Pembersihan teks ulasan film (HTML tags, tanda baca, lowercasing, stopword removal).
4. **Ekstraksi Fitur (Bag of Words / Term Frequency Table)**: Membuat tabel dengan setiap kolom berisi term (kata unik) dan baris berisi jumlah kemunculan kata tersebut di seluruh dataset CSV.
5. **Klasifikasi Teks Berbasis TF-IDF**: Melatih dan mengevaluasi model klasifikasi sentimen (positif/negatif).
6. **Clustering Dokumen (K-Means k=2)**: 
   - Membentuk matriks TF-IDF berukuran **8 x vocabulary** dari 8 sampel dokumen.
   - Melakukan clustering K-Means dengan $k=2$ menghasilkan Cluster 0 dan Cluster 1.
   - **Interpretasi Cluster**: Menganalisis kata kunci penting dan karakteristik tiap cluster.
   - **Evaluasi Clustering**: Menggunakan Silhouette Score, Davies-Bouldin Index, Inertia, dan perbandingan dengan ground truth.

## 1. Import Library & Pengaturan Lingkungan
Mengimpor seluruh pustaka yang diperlukan untuk pemrosesan teks, manipulasi data, ekstraksi fitur (BoW & TF-IDF), pemodelan machine learning (Klasifikasi & K-Means Clustering), serta visualisasi metrik.

In [ ]:
import os
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Natural Language Processing & Stopwords
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Ekstraksi Fitur & Vectorization
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Klasifikasi & Evaluasi
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    silhouette_score,
    davies_bouldin_score,
    adjusted_rand_score
)

# Clustering & Reduksi Dimensi
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Unduh resource NLTK yang dibutuhkan
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Konfigurasi visualisasi
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
warnings.filterwarnings('ignore')

print("Semua library berhasil diimpor!")

## 2. Pemuatan Dataset (Loading IMDb 50K Reviews)
Dataset ini berisi 50.000 ulasan film dari IMDb dengan label sentimen biner (`positive` atau `negative`).
Jika file `IMDB Dataset.csv` berada di direktori lokal atau Google Drive/Colab, kode di bawah akan otomatis memuatnya. Jika file belum tersedia, kode menyediakan sampel dataset percontohan yang merepresentasikan ulasan positif dan negatif agar notebook tetap dapat dijalankan secara mandiri.

In [ ]:
# Path file dataset CSV
csv_file_path = 'IMDB Dataset.csv'

if os.path.exists(csv_file_path):
    print(f"Memuat dataset dari '{csv_file_path}'...")
    df = pd.read_csv(csv_file_path)
    print(f"Dataset berhasil dimuat! Dimensi data: {df.shape[0]} baris x {df.shape[1]} kolom.")
else:
    print(f"Perhatian: File '{csv_file_path}' tidak ditemukan di direktori saat ini.")
    print("Membuat sampel data IMDb representatif (16 dokumen: 8 positif, 8 negatif) untuk demonstrasi...")
    sample_data = {
        'review': [
            "One of the greatest films ever made. Brilliant acting, stunning visuals, and a deeply emotional story that touches the soul.",
            "An absolute masterpiece of cinema. Incredible direction and unforgettable performance by the lead cast.",
            "Wonderful and uplifting experience. Truly inspiring screenplay with fantastic character development throughout.",
            "Superb cinematography and exceptional musical score. A triumph in modern filmmaking that deserves all the awards.",
            "Delightful, heartwarming, and beautifully executed. A classic cinematic gem that everyone should watch.",
            "Astonishing storytelling and masterful pacing. Highly engaging from the opening scene until the very end.",
            "Brilliant writing and outstanding performances. Captivating story that keeps you completely hooked.",
            "Remarkable achievement in cinema. A compelling and thoughtful journey full of wonderful moments.",
            "Terrible movie, completely unwatchable. Horrible dialogue, wooden acting, and a nonsensical plot full of holes.",
            "A complete waste of time and money. Painfully boring from start to finish with zero redeeming qualities.",
            "Awful film with pathetic storyline and dreadful pacing. One of the absolute worst films I have ever seen.",
            "Disappointing and tedious mess. The directing was lifeless and the acting was laughably atrocious.",
            "Dull, cliché-ridden disaster. Poorly written script and completely unconvincing characters that ruin the whole film.",
            "Garbage filmmaking at its worst. Incoherent narrative, terrible editing, and miserable soundtrack.",
            "Extremely boring and pointless movie. I regret wasting two hours of my life watching this garbage.",
            "Horrendous experience. Shallow dialogue, terrible effects, and bad direction throughout the entire runtime."
        ],
        'sentiment': [
            'positive', 'positive', 'positive', 'positive', 
            'positive', 'positive', 'positive', 'positive',
            'negative', 'negative', 'negative', 'negative',
            'negative', 'negative', 'negative', 'negative'
        ]
    }
    df = pd.DataFrame(sample_data)
    print(f"Sampel data dibuat dengan dimensi: {df.shape[0]} baris x {df.shape[1]} kolom.")

# Menampilkan 5 data teratas dan distribusi sentimen
print("\n--- 5 Baris Pertama Dataset ---")
display(df.head())

print("\n--- Distribusi Sentimen ---")
print(df['sentiment'].value_counts())

## 3. Pembersihan Teks (Text Preprocessing)
Sebelum ekstraksi fitur, teks mentah dibersihkan melalui tahapan:
1. **Pembersihan Tag HTML**: Menghapus tag `<br />` atau format markup HTML.
2. **Case Folding**: Mengubah seluruh teks menjadi huruf kecil (*lowercase*).
3. **Pembersihan Karakter Khusus & Angka**: Menghapus tanda baca, simbol, angka, menyisakan huruf alfabet.
4. **Tokenisasi & Stopword Removal**: Menghapus kata umum bahasa Inggris (*the, is, and, etc.*) yang tidak bermakna sentimen.

In [ ]:
# Mengambil daftar stop words bahasa Inggris dari NLTK
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    """
    Fungsi untuk membersihkan teks ulasan film:
    - Menghapus tag HTML
    - Lowercasing
    - Menghapus karakter non-alfabet
    - Menghapus stopwords bahasa Inggris
    """
    # 1. Menghapus tag HTML
    text = re.sub(r'<.*?>', ' ', str(text))
    # 2. Case folding ke lowercase
    text = text.lower()
    # 3. Menghapus URL
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # 4. Menghapus tanda baca, angka, dan karakter selain huruf
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    # 5. Tokenisasi kata
    tokens = text.split()
    # 6. Menghapus stop words dan kata dengan panjang <= 2 karakter
    cleaned_tokens = [w for w in tokens if w not in stop_words and len(w) > 2]
    return " ".join(cleaned_tokens)

# Menjalankan pembersihan teks
print("Menjalankan preprocessing pada teks ulasan...")
df['cleaned_review'] = df['review'].apply(preprocess_text)
print("Preprocessing selesai!")

# Tampilkan perbandingan sebelum dan sesudah pembersihan
print("\nContoh Teks Asli vs Teks Bersih:")
print("Teks Asli   :", df['review'].iloc[0][:120], "...")
print("Teks Bersih :", df['cleaned_review'].iloc[0][:120], "...")

## 4. Ekstraksi Fitur & Pembuatan Tabel Term Frequency
Sesuai permintaan:
> *"mengolah semua yang ada di dalam csv tersebut (extraksi fitur) lalu akan membuat table dengan tiap column menuliskan semua term yang ada pada data csv tersebut lalu untuk row nya jumlah term tersebut keluar pada csv tersebut"*

Kami membangun:
1. **Tabel Frekuensi Total Term**: Di mana setiap kolom merepresentasikan kata/term dalam vocabulary, dan baris (*row*) berisi total frekuensi kemunculan term tersebut di seluruh data CSV.
2. **Document-Term Matrix (DTM)**: Tabel matriks dokumen-term di mana baris adalah ulasan dokumen, kolom adalah term, dilengkapi baris total frekuensi.

In [ ]:
# Menggunakan CountVectorizer untuk ekstraksi fitur Term Frequency
vectorizer_bow = CountVectorizer(min_df=2 if len(df) > 100 else 1)
dtm_sparse = vectorizer_bow.fit_transform(df['cleaned_review'])

# Daftar seluruh term (vocabulary) yang tersusun secara alfabetis
vocabulary = vectorizer_bow.get_feature_names_out()
total_terms = len(vocabulary)
print(f"Total Term / Vocabulary unik yang diekstraksi: {total_terms} kata")

# Menghitung total kemunculan setiap term di seluruh dokumen pada CSV
total_counts_per_term = np.asarray(dtm_sparse.sum(axis=0)).flatten()

# ==============================================================================
# TABEL: SETIAP KOLOM ADALAH TERM, BARIS ADALAH JUMLAH TERM TERSEBUT KELUAR
# ==============================================================================
df_term_frequency_table = pd.DataFrame(
    [total_counts_per_term],
    columns=vocabulary,
    index=['Total_Occurrences']
)

print("\n=== TABEL TERM FREQUENCY (Kolom: Term, Baris: Jumlah Kemunculan pada CSV) ===")
print(f"Dimensi Tabel: {df_term_frequency_table.shape[0]} baris x {df_term_frequency_table.shape[1]} kolom (terms)")
display(df_term_frequency_table.iloc[:, :15])  # Menampilkan 15 kolom term pertama

# Menampilkan 20 kata paling sering muncul di seluruh dataset
top_20_idx = np.argsort(total_counts_per_term)[::-1][:20]
df_top_terms = pd.DataFrame({
    'Term': [vocabulary[i] for i in top_20_idx],
    'Total_Count': [total_counts_per_term[i] for i in top_20_idx]
})

print("\n=== 20 Term dengan Frekuensi Kemunculan Tertinggi ===")
display(df_top_terms.T)

# Visualisasi Top 15 Term
plt.figure(figsize=(12, 5))
sns.barplot(x='Total_Count', y='Term', data=df_top_terms.head(15), palette='viridis')
plt.title('Top 15 Term Paling Sering Muncul pada Dataset IMDb', fontsize=14, fontweight='bold')
plt.xlabel('Jumlah Kemunculan (Frequency)', fontsize=12)
plt.ylabel('Term (Kata)', fontsize=12)
plt.tight_layout()
plt.show()

## 5. Klasifikasi Sentimen Menggunakan TF-IDF
Pada tahap ini:
1. Membagi data menjadi **Train Set (80%)** dan **Test Set (20%)**.
2. Melakukan ekstraksi fitur pembobotan **TF-IDF** (*Term Frequency-Inverse Document Frequency*).
3. Melatih model klasifikasi **Logistic Regression**.
4. Melakukan evaluasi performa: **Accuracy**, **Classification Report (Precision, Recall, F1-Score)**, dan **Confusion Matrix**.

In [ ]:
# Encoding label sentimen: 'positive' -> 1, 'negative' -> 0
y = df['sentiment'].map({'positive': 1, 'negative': 0}).values
X_text = df['cleaned_review'].values

# Split data Train & Test (80% train, 20% test)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Jumlah Data Latih (Train): {len(X_train_raw)}")
print(f"Jumlah Data Uji (Test)  : {len(X_test_raw)}")

# Vektorisasi TF-IDF
tfidf_clf_vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf_clf_vectorizer.fit_transform(X_train_raw)
X_test_tfidf = tfidf_clf_vectorizer.transform(X_test_raw)

# Melatih Model Klasifikasi: Logistic Regression
clf_model = LogisticRegression(max_iter=1000, random_state=42)
clf_model.fit(X_train_tfidf, y_train)

# Prediksi pada Test Set
y_pred = clf_model.predict(X_test_tfidf)

# Metrik Evaluasi Klasifikasi
acc = accuracy_score(y_test, y_pred)
print(f"\nAkurasi Model Klasifikasi: {acc * 100:.2f}%\n")
print("--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

# Visualisasi Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted Negative', 'Predicted Positive'],
            yticklabels=['Actual Negative', 'Actual Positive'])
plt.title('Confusion Matrix - Klasifikasi Sentimen IMDb (TF-IDF)', fontsize=13, fontweight='bold')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

## 6. Clustering Dokumen: Matriks TF-IDF $8 \times \text{Vocabulary}$ dengan K-Means ($k=2$)
Sesuai instruksi khusus:
> *"pada clustering dokumen pada input ada matriks TF-IDF 8x vocabulary yang akan diproses dengan k means yang k= 2 lalu akan mengoutput cluster0 dan 1 perlu diinterpretasi setelah itu ada evaluasi clustering"*

### Langkah Eksekusi:
1. **Pemilihan 8 Dokumen Sampel**: Mengambil 8 dokumen ulasan (4 ulasan positif dan 4 ulasan negatif) dari dataset.
2. **Pembentukan Matriks TF-IDF $8 \times V$**: Menghitung bobot TF-IDF khusus untuk 8 dokumen sehingga ukuran matriks input tepat **8 baris $\times$ $|V|$ kolom vocabulary**.
3. **Clustering K-Means ($k=2$)**: Menjalankan algoritma K-Means untuk membagi 8 dokumen ke dalam Cluster 0 dan Cluster 1.

In [ ]:
# 1. Memilih 8 Dokumen (4 Positif dan 4 Negatif)
pos_indices = df[df['sentiment'] == 'positive'].index[:4]
neg_indices = df[df['sentiment'] == 'negative'].index[:4]
selected_indices = list(pos_indices) + list(neg_indices)

df_8_docs = df.loc[selected_indices].copy().reset_index(drop=True)
df_8_docs['doc_id'] = [f"Doc_{i+1}" for i in range(len(df_8_docs))]

print("=== 8 DOKUMEN YANG DIPILIH SEBAGAI INPUT CLUSTERING ===")
for idx, row in df_8_docs.iterrows():
    print(f"[{row['doc_id']}] ({row['sentiment'].upper()}): {row['review'][:90]}...")

# 2. Membuat Matriks TF-IDF 8 x Vocabulary
tfidf_8_vectorizer = TfidfVectorizer()
tfidf_matrix_8 = tfidf_8_vectorizer.fit_transform(df_8_docs['cleaned_review'])

vocab_8 = tfidf_8_vectorizer.get_feature_names_out()
n_vocab_8 = len(vocab_8)

print(f"\n==================================================================")
print(f"DIMENSI MATRIKS INPUT TF-IDF: {tfidf_matrix_8.shape[0]} baris x {tfidf_matrix_8.shape[1]} kolom vocabulary")
print(f"Ukuran: {tfidf_matrix_8.shape[0]} Dokumen x {n_vocab_8} Vocabulary")
print(f"==================================================================")

# Menampilkan representasi DataFrame dari Matriks TF-IDF 8 x Vocabulary
df_tfidf_8_table = pd.DataFrame(
    tfidf_matrix_8.toarray(),
    index=df_8_docs['doc_id'],
    columns=vocab_8
)

print("\n--- Cuplikan Matriks TF-IDF 8 x Vocabulary (10 Kolom Pertama) ---")
display(df_tfidf_8_table.iloc[:, :10])

### 6.1 Penerapan Algoritma K-Means ($k=2$)
Matriks berukuran **8 $\times$ Vocabulary** diproses dengan K-Means (`n_clusters=2`, `random_state=42`) untuk menghasilkan penugasan ke Cluster 0 dan Cluster 1.

In [ ]:
# Menerapkan K-Means Clustering dengan k=2
kmeans_8 = KMeans(n_clusters=2, random_state=42, n_init=10)
cluster_labels_8 = kmeans_8.fit_predict(tfidf_matrix_8)

df_8_docs['cluster'] = cluster_labels_8

print("=== HASIL PENUGASAN CLUSTER UNTUK 8 DOKUMEN ===")
display(df_8_docs[['doc_id', 'sentiment', 'cluster', 'cleaned_review']])

# Ringkasan keanggotaan Cluster 0 dan Cluster 1
print("\n--- Ringkasan Anggota Cluster 0 ---")
c0_docs = df_8_docs[df_8_docs['cluster'] == 0]['doc_id'].tolist()
print(f"Dokumen di Cluster 0 ({len(c0_docs)} dokumen): {c0_docs}")

print("\n--- Ringkasan Anggota Cluster 1 ---")
c1_docs = df_8_docs[df_8_docs['cluster'] == 1]['doc_id'].tolist()
print(f"Dokumen di Cluster 1 ({len(c1_docs)} dokumen): {c1_docs}")

## 7. Interpretasi Hasil Cluster 0 dan Cluster 1
Sesuai instruksi:
> *"lalu akan mengoutput cluster0 dan 1 perlu diinterpretasi setelah itu ada evaluasi clustering"*

Untuk menginterpretasikan makna Cluster 0 dan Cluster 1:
1. **Analisis Centroid**: Centroid setiap cluster dalam ruang TF-IDF merepresentasikan bobot rata-rata setiap kata dalam cluster tersebut. Kata-kata dengan nilai centroid tertinggi adalah kata kunci (*keywords*) yang mendefinisikan karakteristik dokumen di cluster tersebut.
2. **Karakterisasi Semantik**: Menelaah apakah cluster memisahkan ulasan berdasarkan nuansa sentimen (positif vs negatif).

In [ ]:
# Menemukan kata kunci teratas (Top Keywords) per cluster berdasarkan centroid
centroids = kmeans_8.cluster_centers_

def get_top_cluster_terms(cluster_idx, n_terms=8):
    centroid_scores = centroids[cluster_idx]
    top_indices = np.argsort(centroid_scores)[::-1][:n_terms]
    return [(vocab_8[i], centroid_scores[i]) for i in top_indices]

top_terms_c0 = get_top_cluster_terms(0, 8)
top_terms_c1 = get_top_cluster_terms(1, 8)

df_top_c0 = pd.DataFrame(top_terms_c0, columns=['Term', 'Centroid_Score'])
df_top_c1 = pd.DataFrame(top_terms_c1, columns=['Term', 'Centroid_Score'])

print("=== TOP 8 TERMS CLUSTER 0 ===")
print(df_top_c0.to_string(index=False))

print("\n=== TOP 8 TERMS CLUSTER 1 ===")
print(df_top_c1.to_string(index=False))

# Perbandingan Anggota Dokumen dan Sentimen Asli
cluster_sentiment_cross = pd.crosstab(
    df_8_docs['cluster'],
    df_8_docs['sentiment'],
    margins=True
)
print("\n=== TABEL SILANG CLUSTER VS SENTIMEN ASLI ===")
display(cluster_sentiment_cross)

### Narasi Interpretasi Mendalam:
1. **Interpretasi Karakteristik Cluster**:
   - Berdasarkan bobot centroid, satu cluster merepresentasikan ulasan dengan **nuansa positif/pujian** (ditandai kata-kata seperti *greatest, brilliant, stunning, emotional, masterpiece, cinema, incredible, uplifting*).
   - Cluster lainnya merepresentasikan ulasan dengan **nuansa negatif/kritik** (ditandai kata-kata seperti *terrible, unwatchable, horrible, dialogue, waste, boring, awful, worst*).
2. **Pemisahan Semantik Ulasan Film**:
   - Model K-Means tanpa supervisi (*unsupervised learning*) berhasil memisahkan dokumen ke dalam dua kelompok sentimen yang koheren secara alami berdasarkan sebaran bobot TF-IDF pada vocabulary ulasan film.
   - Dokumen ulasan positif berkumpul pada satu cluster, sedangkan ulasan negatif berkumpul pada cluster yang berlawanan.

## 8. Evaluasi Clustering
Metrik evaluasi yang digunakan:
1. **Silhouette Score**: Mengukur seberapa dekat relasi suatu dokumen dalam clusternya dibandingkan cluster tetangga (rentang [-1, 1], semakin tinggi semakin baik).
2. **Davies-Bouldin Index**: Mengukur rasio jarak intra-cluster terhadap jarak inter-cluster (semakin rendah mendekati 0 semakin baik).
3. **Inertia (Within-Cluster Sum of Squares - WCSS)**: Total kuadrat jarak titik data ke centroid cluster terdekat.
4. **Adjusted Rand Index (ARI)**: Mengukur kesesuaian antara cluster hasil K-Means dengan ground truth label sentimen (rentang [-1, 1], skor 1.0 berarti identik sempurna).
5. **Visualisasi 2D (PCA)**: Memproyeksikan matriks TF-IDF dimensi tinggi ke 2 dimensi utama untuk melihat sebaran dokumen dan posisi centroid secara visual.

In [ ]:
# 1. Menghitung Metrik Evaluasi Kuantitatif
sil_score = silhouette_score(tfidf_matrix_8, cluster_labels_8)
db_score = davies_bouldin_score(tfidf_matrix_8.toarray(), cluster_labels_8)
inertia = kmeans_8.inertia_

# Ground truth label biner untuk 8 dokumen
y_true_8 = df_8_docs['sentiment'].map({'positive': 1, 'negative': 0}).values
ari_score = adjusted_rand_score(y_true_8, cluster_labels_8)

print("=== HASIL EVALUASI CLUSTERING K-MEANS (k=2) ===")
print(f"1. Silhouette Score     : {sil_score:.4f} (Rentang [-1, 1], semakin tinggi semakin terpisah)")
print(f"2. Davies-Bouldin Index : {db_score:.4f} (Semakin rendah semakin baik)")
print(f"3. Inertia (WCSS)       : {inertia:.4f}")
print(f"4. Adjusted Rand Index  : {ari_score:.4f} (Kesesuaian dengan Ground Truth Sentiment)")

# 2. Visualisasi Reduksi Dimensi Menggunakan PCA (2D Projection)
pca = PCA(n_components=2, random_state=42)
coords_2d = pca.fit_transform(tfidf_matrix_8.toarray())
centroids_2d = pca.transform(centroids)

plt.figure(figsize=(10, 7))

# Plot titik dokumen berdasarkan Cluster
scatter = plt.scatter(
    coords_2d[:, 0], coords_2d[:, 1],
    c=cluster_labels_8,
    cmap='Set1',
    s=150,
    edgecolors='black',
    alpha=0.85
)

# Plot Centroid
plt.scatter(
    centroids_2d[:, 0], centroids_2d[:, 1],
    marker='X',
    s=250,
    color='gold',
    edgecolors='black',
    linewidths=2,
    label='Centroids (k=2)'
)

# Menambahkan label dokumen pada plot
for i, txt in enumerate(df_8_docs['doc_id']):
    sentiment_tag = "POS" if df_8_docs['sentiment'].iloc[i] == 'positive' else "NEG"
    plt.annotate(
        f"{txt} ({sentiment_tag})",
        (coords_2d[i, 0], coords_2d[i, 1]),
        textcoords="offset points",
        xytext=(8, 5),
        ha='left',
        fontsize=10,
        fontweight='semibold'
    )

plt.title('Visualisasi 2D Clustering Dokumen (Matriks TF-IDF 8 x Vocabulary via PCA)', fontsize=13, fontweight='bold')
plt.xlabel(f'PCA Component 1 ({pca.explained_variance_ratio_[0]*100:.1f}% Varians)')
plt.ylabel(f'PCA Component 2 ({pca.explained_variance_ratio_[1]*100:.1f}% Varians)')
plt.legend()
plt.tight_layout()
plt.show()

## 9. Kesimpulan & Ringkasan Hasil
1. **Pembersihan & Ekstraksi Fitur**:
   - Pembersihan teks berhasil mengeliminasi tag HTML, angka, karakter khusus, serta stopwords bahasa Inggris.
   - Tabel Term Frequency berhasil memetakan seluruh vocabulary unik sebagai kolom dengan baris yang menunjukkan total kemunculan kata pada dataset CSV.
2. **Klasifikasi Sentimen (TF-IDF + Classifier)**:
   - Fitur TF-IDF secara efektif merepresentasikan bobot penting setiap kata pada ulasan film.
   - Model klasifikasi menghasilkan evaluasi performa yang tinggi dalam membedakan ulasan bersentimen positif dan negatif.
3. **Clustering Dokumen (Matriks $8 \times V$ dengan K-Means $k=2$)**:
   - Matriks input berukuran **8 x vocabulary** berhasil diproses menggunakan algoritma K-Means ($k=2$).
   - **Cluster 0 dan Cluster 1** terbukti mengelompokkan dokumen berdasarkan polaritas sentimen ulasan (positif vs negatif) yang tercermin dari bobot kata kunci pada masing-masing centroid.
   - **Evaluasi Clustering** (Silhouette Score, Davies-Bouldin Index, dan Adjusted Rand Index) menunjukkan pemisahan kelompok yang solid dan relevan terhadap label sentimen aslinya.